# HEIM — benchmark inference trên 2.000 ảnh

Chọn **Runtime → Change runtime type → T4 GPU** (hoặc GPU khác), rồi chạy từng ô.
Notebook tự tải ảnh có caption từ COCO val2017 và chấm bằng CLIP ViT-L/14 + LAION aesthetic head.
Chỉ cần upload notebook này; các script đã được nhúng bên trong. Không cần API key.

**Kết quả chính:** thời gian toàn pipeline và ảnh/giây. **Kết quả bổ sung:** thời gian inference.
Mặc định batch size 1, warm-up 20 ảnh, 3 lượt đo × 2.000 ảnh.
Trọng số, dữ liệu, cài đặt và warm-up được tải/chạy trước khoảng đo.

**Dữ liệu ảnh:** lưu lâu dài tại `MyDrive/test_runtime` sau khi kết nối Google Drive. Bộ ảnh 1024 × 1024 được chép sang `/content` trước khi đo.


In [1]:
from pathlib import Path
import datetime, json, os, subprocess, sys

WORK = Path('/content/heim_runtime')
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
COUNT = 2000
SEED = 2021
REPEATS = 3
WARMUP = 20
RUN_NAME = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
RESULTS = WORK / 'results' / RUN_NAME
print('Working directory:', WORK)
print('Result directory:', RESULTS)


Working directory: /content/heim_runtime
Result directory: /content/heim_runtime/results/20261116T050921_443913Z


## 1. Cài phụ thuộc

Dùng PyTorch/torchvision của Colab. OpenAI CLIP được cố định theo commit của dự án.
Nếu Colab yêu cầu khởi động lại runtime sau cài đặt, thực hiện rồi chạy lại từ ô cấu hình.


In [2]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'Pillow', 'ftfy', 'regex', 'tqdm', 'setuptools<81', 'wheel'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation',
                'git+https://github.com/openai/CLIP.git@a1d071733d7111c9c014f024669f959182114e33'], check=True)
import torch
assert torch.cuda.is_available(), 'Hãy chọn GPU runtime trước khi benchmark.'
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name())


PyTorch: 2.14.1+cu130
GPU: Tesla T4


## 2. Ghi script từ nội dung đã nhúng
Ô này chỉ tạo file, chưa tải ảnh hoặc chạy mô hình.


In [3]:
(WORK / 'download_data.py').write_text('"""Download a deterministic COCO validation subset; never generate images."""\nimport argparse\nimport concurrent.futures\nimport hashlib\nimport json\nimport random\nimport time\nimport urllib.request\nimport zipfile\nfrom pathlib import Path\n\nfrom PIL import Image\n\nANNOTATIONS = \'https://images.cocodataset.org/annotations/annotations_trainval2017.zip\'\n\n\ndef fetch(url, target):\n    part = target.with_suffix(target.suffix + \'.part\')\n    for attempt in range(4):\n        try:\n            request = urllib.request.Request(url, headers={\'User-Agent\': \'HEIM-runtime-test/1.0\'})\n            with urllib.request.urlopen(request, timeout=90) as src, part.open(\'wb\') as dst:\n                while chunk := src.read(1024 * 1024):\n                    dst.write(chunk)\n            part.replace(target)\n            return\n        except Exception:\n            part.unlink(missing_ok=True)\n            if attempt == 3:\n                raise\n            time.sleep(2 ** attempt)\n\n\ndef inspect(path):\n    with Image.open(path) as image:\n        image.load()\n        width, height = image.size\n        if width < 1 or height < 1:\n            raise ValueError(\'Empty image\')\n    return width, height, hashlib.sha256(path.read_bytes()).hexdigest()\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--output\', type=Path, default=Path(\'data\'))\n    parser.add_argument(\'--count\', type=int, default=2000)\n    parser.add_argument(\'--seed\', type=int, default=2021)\n    parser.add_argument(\'--workers\', type=int, default=8)\n    args = parser.parse_args()\n    if not 1 <= args.count <= 5000 or args.workers < 1:\n        parser.error(\'count must be 1..5000 and workers must be positive\')\n    args.output.mkdir(parents=True, exist_ok=True)\n    images_dir = args.output / \'images\'\n    images_dir.mkdir(exist_ok=True)\n    annotation = args.output / \'captions_val2017.json\'\n    if not annotation.exists():\n        archive = args.output / \'annotations_trainval2017.zip\'\n        if not archive.exists():\n            print(\'Downloading official annotations (~253 MB)\', flush=True)\n            fetch(ANNOTATIONS, archive)\n        with zipfile.ZipFile(archive) as source:\n            annotation.write_bytes(source.read(\'annotations/captions_val2017.json\'))\n    source = json.loads(annotation.read_text())\n    captions = {}\n    for row in sorted(source[\'annotations\'], key=lambda r: r[\'id\']):\n        captions.setdefault(row[\'image_id\'], row[\'caption\'])\n    candidates = sorted(source[\'images\'], key=lambda r: r[\'id\'])\n    random.Random(args.seed).shuffle(candidates)\n\n    def download(row):\n        path = images_dir / row[\'file_name\']\n        url = \'https://images.cocodataset.org/val2017/\' + row[\'file_name\']\n        try:\n            if path.exists():\n                try:\n                    inspect(path)\n                except Exception:\n                    path.unlink()\n            if not path.exists():\n                fetch(url, path)\n            width, height, digest = inspect(path)\n            return {\'id\': row[\'id\'], \'image\': \'images/\' + row[\'file_name\'],\n                    \'caption\': captions[row[\'id\']], \'url\': url,\n                    \'width\': width, \'height\': height, \'sha256\': digest}\n        except Exception as error:\n            return {\'id\': row[\'id\'], \'error\': str(error), \'url\': url}\n\n    records, errors, hashes = [], [], set()\n    with concurrent.futures.ThreadPoolExecutor(args.workers) as pool:\n        # Map preserves the seeded order, independent of download completion order.\n        # Download only the number still needed; replace failed or duplicate files.\n        position = 0\n        while len(records) < args.count and position < len(candidates):\n            needed = args.count - len(records)\n            batch = candidates[position:position + needed]\n            position += len(batch)\n            for result in pool.map(download, batch):\n                if \'error\' in result:\n                    errors.append(result)\n                elif result[\'sha256\'] not in hashes:\n                    records.append(result)\n                    hashes.add(result[\'sha256\'])\n            print(f\'Valid unique images: {len(records)}/{args.count}\', flush=True)\n    (args.output / \'download_errors.json\').write_text(json.dumps(errors, indent=2))\n    if len(records) != args.count:\n        raise RuntimeError(f\'Only {len(records)} valid images; retry to resume downloads\')\n    manifest = {\'dataset\': \'COCO val2017\', \'seed\': args.seed, \'count\': len(records),\n                \'caption_policy\': \'lowest annotation ID for each image\',\n                \'records\': records}\n    (args.output / \'manifest.json\').write_text(json.dumps(manifest, indent=2), encoding=\'utf-8\')\n    print(\'Ready:\', args.output / \'manifest.json\')\n\n\nif __name__ == \'__main__\':\n    main()\n', encoding='utf-8')
(WORK / 'benchmark.py').write_text('"""Batch-1 benchmark matching HEIM pilot CLIP + aesthetic scoring."""\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport platform\nimport statistics\nimport time\nfrom pathlib import Path\n\nimport torch\nfrom PIL import Image\n\n\nclass Scorer:\n    def __init__(self):\n        import clip\n        from urllib.request import urlretrieve\n        self.clip = clip\n        self.device = \'cuda\' if torch.cuda.is_available() else \'cpu\'\n        self.model, self.preprocess = clip.load(\'ViT-L/14\', device=self.device)\n        self.model.eval()\n        self.head = torch.nn.Linear(768, 1).to(self.device)\n        weights = Path.home() / \'.cache/heim_pilot/sa_0_4_vit_l_14_linear.pth\'\n        weights.parent.mkdir(parents=True, exist_ok=True)\n        if not weights.exists():\n            urlretrieve(\'https://raw.githubusercontent.com/LAION-AI/aesthetic-predictor/main/sa_0_4_vit_l_14_linear.pth\', weights)\n        self.head.load_state_dict(torch.load(weights, map_location=self.device, weights_only=True))\n        self.head.eval()\n        self.cache = {}\n\n    def sync(self):\n        if self.device == \'cuda\':\n            torch.cuda.synchronize()\n\n    @torch.no_grad()\n    def score(self, path, caption):\n        # Decode, preprocess, tokenize and host-to-device copies are outside the\n        # inference interval but inside the enclosing pipeline timer.\n        with Image.open(path) as image:\n            tensor = self.preprocess(image.convert(\'RGB\')).unsqueeze(0).to(self.device)\n        tokens = None\n        if caption not in self.cache:\n            tokens = self.clip.tokenize([caption], truncate=True).to(self.device)\n        if self.device == \'cuda\':\n            start, end = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)\n            start.record()\n        else:\n            begin = time.perf_counter()\n        features = self.model.encode_image(tensor).float()\n        features = features / features.norm(dim=-1, keepdim=True)\n        if tokens is not None:\n            text = self.model.encode_text(tokens).float()\n            self.cache[caption] = text / text.norm(dim=-1, keepdim=True)\n        similarity = (features @ self.cache[caption].T).clamp(min=0) * 100\n        aesthetic = self.head(features)\n        if self.device == \'cuda\':\n            end.record()\n            end.synchronize()\n            seconds = start.elapsed_time(end) / 1000\n        else:\n            seconds = time.perf_counter() - begin\n        return seconds, similarity.item(), aesthetic.item()\n\n\ndef run_pass(scorer, records, root, repeat):\n    rows = []\n    scorer.sync()\n    start = time.perf_counter()\n    for record in records:\n        item_start = time.perf_counter()\n        # Fail explicitly on any scoring error; never silently score < N images.\n        seconds, clip_score, aesthetic_score = scorer.score(root / record[\'image\'], record[\'caption\'])\n        rows.append({\'repeat\': repeat, \'id\': record[\'id\'], \'inference_seconds\': seconds,\n                     \'pipeline_seconds\': time.perf_counter() - item_start,\n                     \'clip_score\': clip_score, \'aesthetics_score\': aesthetic_score})\n    scorer.sync()\n    elapsed = time.perf_counter() - start\n    model_seconds = sum(r[\'inference_seconds\'] for r in rows)\n    summary = {\'repeat\': repeat, \'images\': len(rows), \'pipeline_seconds\': elapsed,\n               \'pipeline_images_per_second\': len(rows) / elapsed,\n               \'pipeline_ms_per_image\': 1000 * elapsed / len(rows),\n               \'inference_seconds\': model_seconds,\n               \'inference_images_per_second\': len(rows) / model_seconds,\n               \'inference_ms_per_image\': 1000 * model_seconds / len(rows)}\n    return rows, summary\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--manifest\', type=Path, default=Path(\'data/manifest.json\'))\n    parser.add_argument(\'--output\', type=Path, default=Path(\'results\'))\n    parser.add_argument(\'--expected-count\', type=int, default=2000)\n    parser.add_argument(\'--repeats\', type=int, default=3)\n    parser.add_argument(\'--warmup\', type=int, default=20)\n    args = parser.parse_args()\n    if args.repeats < 1 or args.warmup < 0 or args.expected_count < 1:\n        parser.error(\'Invalid repeats, warmup or expected-count\')\n    raw = args.manifest.read_bytes()\n    manifest = json.loads(raw)\n    records = manifest[\'records\']\n    if len(records) != args.expected_count or manifest[\'count\'] != len(records):\n        raise ValueError(\'Manifest count differs from expected-count\')\n    if len({r[\'id\'] for r in records}) != len(records):\n        raise ValueError(\'Duplicate image IDs\')\n    root = args.manifest.parent.resolve()\n    print(\'Validating all files before timing...\', flush=True)\n    for record in records:\n        path = (root / record[\'image\']).resolve()\n        if not path.is_relative_to(root):\n            raise ValueError(\'Image path outside dataset\')\n        if hashlib.sha256(path.read_bytes()).hexdigest() != record[\'sha256\']:\n            raise ValueError(f\'Image checksum mismatch: {path}\')\n        with Image.open(path) as image:\n            image.load()\n    scorer = Scorer()\n    print(f\'Device: {scorer.device}; warming up {args.warmup} images\', flush=True)\n    for index in range(args.warmup):\n        record = records[index % len(records)]\n        scorer.score(root / record[\'image\'], record[\'caption\'])\n    scorer.sync()\n    args.output.mkdir(parents=True, exist_ok=True)\n    runs = []\n    for repeat in range(1, args.repeats + 1):\n        scorer.cache.clear()  # Empty text cache at the start of every measured pass.\n        if scorer.device == \'cuda\':\n            torch.cuda.reset_peak_memory_stats()\n        rows, summary = run_pass(scorer, records, root, repeat)\n        summary[\'peak_gpu_allocated_bytes\'] = torch.cuda.max_memory_allocated() if scorer.device == \'cuda\' else None\n        with (args.output / f\'per_image_run_{repeat}.csv\').open(\'w\', newline=\'\') as stream:\n            writer = csv.DictWriter(stream, fieldnames=list(rows[0]))\n            writer.writeheader()\n            writer.writerows(rows)\n        runs.append(summary)\n        print(json.dumps(summary), flush=True)\n    report = {\'configuration\': {\'batch_size\': 1, \'model\': \'CLIP ViT-L/14\',\n              \'aesthetic_head\': \'LAION sa_0_4_vit_l_14_linear\',\n              \'text_cache\': \'empty each pass; reused for identical captions within pass\',\n              \'warmup_images\': args.warmup, \'repeats\': args.repeats,\n              \'manifest_sha256\': hashlib.sha256(raw).hexdigest()},\n              \'environment\': {\'python\': platform.python_version(), \'torch\': torch.__version__,\n              \'cuda\': torch.version.cuda, \'device\': scorer.device,\n              \'gpu\': torch.cuda.get_device_name() if scorer.device == \'cuda\' else None,\n              \'clip_dtype\': str(next(scorer.model.parameters()).dtype)},\n              \'timing\': {\'inference\': \'image encoding, uncached text encoding, normalization, similarity and aesthetic head; CUDA events on GPU, perf_counter on CPU\',\n              \'pipeline\': \'local file read/decode, preprocessing, tokenization, transfers, inference, scalar results, timing instrumentation and loop overhead\',\n              \'excluded\': \'network downloads, setup, model loading, validation, warmup and output file writes\',\n              \'filesystem_cache\': \'warm OS cache: validation reads all images before timing; sequential batch 1 with GPU sync per image\'},\n              \'runs\': runs,\n              \'median\': {key: statistics.median(run[key] for run in runs)\n                         for key in runs[0] if key not in (\'repeat\', \'peak_gpu_allocated_bytes\')},\n              \'pipeline_seconds_min\': min(r[\'pipeline_seconds\'] for r in runs),\n              \'pipeline_seconds_max\': max(r[\'pipeline_seconds\'] for r in runs)}\n    (args.output / \'report.json\').write_text(json.dumps(report, indent=2))\n    with (args.output / \'runs.csv\').open(\'w\', newline=\'\') as stream:\n        writer = csv.DictWriter(stream, fieldnames=list(runs[0]))\n        writer.writeheader()\n        writer.writerows(runs)\n    print(\'Median:\', json.dumps(report[\'median\'], indent=2))\n\n\nif __name__ == \'__main__\':\n    main()\n', encoding='utf-8')
print('Scripts ready')


Scripts ready


## 3. Kết nối Drive và chuẩn bị ảnh

Chạy ô tiếp theo rồi cấp quyền kết nối Google Drive. Ảnh gốc và ảnh 1024 × 1024
được lưu tại `MyDrive/test_runtime/data` và `MyDrive/test_runtime/data_1024`.
Ảnh đã lưu hợp lệ được tái sử dụng; có thể chạy lại nếu bị ngắt. Nếu dữ liệu của
phiên cũ còn trong `/content/heim_runtime`, notebook sẽ sao chép phần còn thiếu
lên Drive trước. Ảnh 1024 × 1024 được kéo trực tiếp thành hình vuông và lưu JPEG.

Sau đó notebook chép bộ 1024 × 1024 sang `/content/heim_runtime/data_1024`
để chạy benchmark trên ổ cục bộ. Các bước tải, resize và sao chép không tính giờ.


In [4]:
script = WORK / 'download_data.py'
source = script.read_text(encoding='utf-8')
source = source.replace(
    'https://images.cocodataset.org',
    'https://s3.amazonaws.com/images.cocodataset.org'
)
script.write_text(source, encoding='utf-8')
print('Đã sửa URL tải annotation và ảnh.')

Đã sửa URL tải annotation và ảnh.


In [ ]:
import hashlib
import json
import shutil
from pathlib import Path
from PIL import Image
from google.colab import drive

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/test_runtime')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
source_dir = DRIVE_ROOT / 'data'
target_dir = DRIVE_ROOT / 'data_1024'
local_dir = WORK / 'data_1024'

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

# Preserve previously downloaded originals if this Colab runtime still has them.
old_source = WORK / 'data'
if (old_source / 'images').is_dir():
    source_dir.mkdir(parents=True, exist_ok=True)
    for path in old_source.rglob('*'):
        if path.is_file() and not path.name.endswith('.part'):
            destination = source_dir / path.relative_to(old_source)
            if not destination.exists():
                destination.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(path, destination)
    print('Đã sao chép dữ liệu ảnh gốc còn lại trong phiên Colab lên Drive.', flush=True)

# download_data.py resumes and validates existing originals before reuse.
script = WORK / 'download_data.py'
assert script.is_file(), 'Hãy chạy ô 2 để tạo script trước.'
subprocess.run(
    [sys.executable, '-u', str(script), '--output', str(source_dir),
     '--count', str(COUNT), '--seed', str(SEED), '--workers', '8'],
    stderr=subprocess.STDOUT, check=True
)
original = json.loads((source_dir / 'manifest.json').read_text(encoding='utf-8'))
images_dir = target_dir / 'images'
images_dir.mkdir(parents=True, exist_ok=True)
existing_manifest = target_dir / 'manifest.json'
old_records = {}
if existing_manifest.exists():
    try:
        previous = json.loads(existing_manifest.read_text(encoding='utf-8'))
        old_records = {row['id']: row for row in previous['records']}
    except (ValueError, KeyError, TypeError):
        print('Manifest ảnh 1024 cũ không hợp lệ; sẽ kiểm tra và tạo lại.', flush=True)

converted = []
for index, row in enumerate(original['records'], 1):
    source_path = source_dir / row['image']
    output_path = images_dir / f"{row['id']}.jpg"
    old = old_records.get(row['id'], {})
    valid = False
    if (old.get('original_sha256') == row['sha256'] and
            old.get('width') == 1024 and old.get('height') == 1024 and
            old.get('image') == f"images/{output_path.name}" and output_path.is_file()):
        try:
            with Image.open(output_path) as im:
                im.load()
                valid = im.size == (1024, 1024) and sha256(output_path) == old.get('sha256')
        except (OSError, ValueError):
            valid = False
    if not valid:
        partial = output_path.with_suffix('.jpg.part')
        with Image.open(source_path) as im:
            im.convert('RGB').resize((1024, 1024), Image.Resampling.LANCZOS).save(
                partial, format='JPEG', quality=95)
        partial.replace(output_path)
    converted.append({**row, 'original_width': row['width'], 'original_height': row['height'],
                      'original_sha256': row['sha256'], 'width': 1024, 'height': 1024,
                      'image': f'images/{output_path.name}', 'sha256': sha256(output_path)})
    if index % 100 == 0 or index == len(original['records']):
        print(f'Ảnh 1024 × 1024 trên Drive: {index}/{len(original["records"])}', flush=True)

manifest = {**original, 'records': converted,
            'image_processing': 'Resize trực tiếp 1024x1024; JPEG quality 95'}
target_dir.mkdir(parents=True, exist_ok=True)
temp_manifest = target_dir / 'manifest.json.part'
temp_manifest.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')
temp_manifest.replace(existing_manifest)

# Stage only manifest-referenced images locally before any benchmark timer starts.
local_images = local_dir / 'images'
local_images.mkdir(parents=True, exist_ok=True)
for index, row in enumerate(converted, 1):
    local_path = local_dir / row['image']
    if not local_path.exists() or sha256(local_path) != row['sha256']:
        partial = local_path.with_suffix('.jpg.part')
        shutil.copyfile(target_dir / row['image'], partial)
        if sha256(partial) != row['sha256']:
            partial.unlink(missing_ok=True)
            raise ValueError(f'Lỗi sao chép: {row["image"]}')
        partial.replace(local_path)
    if index % 100 == 0 or index == len(converted):
        print(f'Ảnh sẵn sàng trên Colab: {index}/{len(converted)}', flush=True)
shutil.copyfile(existing_manifest, local_dir / 'manifest.json')
print('Ảnh gốc trên Drive:', source_dir)
print('Ảnh 1024 trên Drive:', target_dir)
print('Manifest benchmark:', local_dir / 'manifest.json')


In [ ]:
print('Dữ liệu đã chuẩn bị. Chạy ô benchmark trong mục 4 bên dưới.')


In [ ]:
import json
from collections import Counter

manifest = json.loads((DRIVE_ROOT / 'data/manifest.json').read_text(encoding='utf-8'))
sizes = Counter((r['width'], r['height']) for r in manifest['records'])
print('Số kích thước ảnh gốc khác nhau:', len(sizes))
for (width, height), count in sizes.most_common(10):
    print(f'{width} × {height}: {count} ảnh')


## 4. Đo tốc độ

Mỗi lượt chạy đúng COUNT ảnh. Thời gian **inference** gồm image encoder, text encoder khi caption chưa có trong cache, chuẩn hóa đặc trưng, CLIP similarity và aesthetic head. GPU dùng CUDA events; cache caption được xóa đầu mỗi lượt và tái sử dụng trong lượt, giống cách scorer hiện tại hoạt động.

**Toàn pipeline** gồm đọc/giải mã ảnh, tiền xử lý, token hóa, truyền dữ liệu CPU↔GPU, inference, lấy điểm số, vòng lặp và phần đo thời gian. Đo tuần tự, đồng bộ GPU từng ảnh. CUDA event đo khoảng thời gian trên luồng GPU, có thể bao gồm khoảng chờ giữa các kernel; không phải tổng thời gian kernel thuần túy.

File được kiểm tra trước khi đo, nên đây là phép đo với cache hệ điều hành đã được làm nóng. Không đo tốc độ đọc ổ đĩa lạnh. Kết quả phụ thuộc GPU, CPU, kích thước ảnh gốc và số caption khác nhau.

Không tính tải dữ liệu/trọng số, khởi tạo mô hình, kiểm tra dữ liệu, warm-up hoặc ghi file báo cáo. Kết quả là tốc độ chấm CLIP + aesthetic của pilot; các tiêu chí cần người chấm không nằm trong phép đo này.

Ảnh được đọc từ bản sao 1024 × 1024 trên ổ cục bộ `/content`; CLIP tiếp tục tiền xử lý về 224 × 224 pixel.


In [ ]:
subprocess.run([sys.executable, '-u', str(WORK / 'benchmark.py'),
                '--manifest', str(WORK / 'data_1024/manifest.json'), '--output', str(RESULTS),
                '--expected-count', str(COUNT), '--repeats', str(REPEATS),
                '--warmup', str(WARMUP)], stderr=subprocess.STDOUT, check=True)


## 5. Xem và tải kết quả
ZIP chứa report.json, runs.csv, CSV điểm/timing từng ảnh và thông tin môi trường.


In [ ]:
import shutil
from google.colab import files
from IPython.display import display
import pandas as pd

report = json.loads((RESULTS / 'report.json').read_text())
display(pd.read_csv(RESULTS / 'runs.csv'))
print('Trung vị qua các lượt:')
print(json.dumps(report['median'], indent=2))
print('Môi trường:')
print(json.dumps(report['environment'], indent=2))
shutil.copy2(WORK / 'data_1024/manifest.json', RESULTS / 'manifest.json')
with (RESULTS / 'pip_freeze.txt').open('w') as stream:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=stream, check=True)
archive = shutil.make_archive(str(WORK / ('heim_runtime_' + RUN_NAME)), 'zip', RESULTS)
files.download(archive)
